# Anti-UAV YOLO26m Training — Run 2
**Model:** yolo26m | **Hardware:** Colab T4 | **Dataset:** Anti-UAV merged (Bird/Drone/UAV)

**Before running:**
1. Upload `backup_merged_dataset.tar.gz` to your Google Drive root
2. Set Runtime → Change runtime type → T4 GPU
3. Run All

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
import os
print('Drive mounted')
# Check GPU
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
print('GPU:', result.stdout.strip())


In [ ]:
# Find and extract dataset from Drive
import tarfile, os, yaml

# Search for the backup in Drive
tar_path = None
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for f in files:
        if f == 'backup_merged_dataset.tar.gz':
            tar_path = os.path.join(root, f)
            break
    if tar_path:
        break

if tar_path is None:
    raise FileNotFoundError(
        'backup_merged_dataset.tar.gz not found in Google Drive.\n'
        'Please upload datasets/backup_merged_dataset.tar.gz to your Drive root.'
    )

print(f'Found: {tar_path}')
extract_dir = '/content/dataset'
os.makedirs(extract_dir, exist_ok=True)
print('Extracting (this takes ~2 minutes)...')
with tarfile.open(tar_path) as tf:
    tf.extractall(extract_dir)

data_yaml = os.path.join(extract_dir, 'merged_dataset', 'data.yaml')
print(f'data.yaml found: {os.path.exists(data_yaml)}')

# Fix paths
base = os.path.join(extract_dir, 'merged_dataset')
with open(data_yaml) as f:
    cfg = yaml.safe_load(f)
cfg['path'] = base
cfg['train'] = os.path.join(base, 'train', 'images')
cfg['val'] = os.path.join(base, 'val', 'images')
cfg['test'] = os.path.join(base, 'test', 'images')
with open(data_yaml, 'w') as f:
    yaml.dump(cfg, f)
print('Paths updated:', cfg)


In [ ]:
# Install ultralytics
import subprocess
subprocess.run(['pip', 'install', '-q', 'ultralytics>=8.4.0'], check=True)
from ultralytics import YOLO
print('ultralytics ready')


In [ ]:
# Run training — yolo26m on T4
from ultralytics import YOLO
model = YOLO('yolo26m.pt')
results = model.train(
    data=data_yaml,
    imgsz=640,
    batch=32,
    epochs=100,
    optimizer='MuSGD',
    lr0=0.01,
    weight_decay=0.0005,
    amp=True,
    device='0',
    mosaic=1.0,
    mixup=0.05,
    copy_paste=0.5,
    hsv_h=0.02,
    hsv_s=0.7,
    hsv_v=0.5,
    degrees=20.0,
    translate=0.15,
    scale=0.8,
    flipud=0.3,
    fliplr=0.5,
    project='/content/runs',
    name='anti_uav_run2_yolo26m',
)
print('Training complete')
print(f'Results saved to: {results.save_dir}')


In [ ]:
# Archive full run directory and save to Drive
import zipfile, os, shutil

runs_dir = '/content/runs/anti_uav_run2_yolo26m'
archive_local = '/content/anti_uav_run2_yolo26m_full.zip'
archive_drive = '/content/drive/MyDrive/anti_uav_run2_yolo26m_full.zip'

print('Archiving full run directory...')
with zipfile.ZipFile(archive_local, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(runs_dir):
        for file in files:
            filepath = os.path.join(root, file)
            arcname = os.path.relpath(filepath, '/content')
            zf.write(filepath, arcname)
            size = os.path.getsize(filepath) / 1024 / 1024
            print(f'  {arcname} ({size:.1f} MB)')

# Copy to Drive for persistence
shutil.copy2(archive_local, archive_drive)
size = os.path.getsize(archive_drive) / 1024 / 1024
print(f'\nSaved to Drive: {archive_drive} ({size:.1f} MB)')
print('Done — download from Google Drive or the Files panel')
